In [1]:
import pandas as pd
import random
import csv
import numpy as np
import os
import umap


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Testing for Best UMAP Parameters

In [2]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

embeddings_data = np.load('datasets/bible_embeddings.npz')
embeddings = embeddings_data['embeddings']

def neighborhood_preservation(high_dim, low_dim, k=10):
    """
    Fraction of high-dimensional neighbors preserved in low-dimensional embedding.
    """
    nbrs_high = NearestNeighbors(n_neighbors=k).fit(high_dim)
    _, idx_high = nbrs_high.kneighbors(high_dim)

    nbrs_low = NearestNeighbors(n_neighbors=k).fit(low_dim)
    _, idx_low = nbrs_low.kneighbors(low_dim)

    preserved = [len(set(idx_high[i]) & set(idx_low[i])) / k for i in range(len(high_dim))]
    return np.mean(preserved)

n_neighbors_options = [10, 15, 20, 25, 30]
min_dist_options = [0.05, 0.1, 0.15, 0.2, 0.3]

best_score = -1
best_params = {}

for n_neighbors in n_neighbors_options:
    for min_dist in min_dist_options:
        reducer = umap.UMAP(
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            n_components=3,
            metric='cosine',
            random_state=42
        )
        embedding_3d = reducer.fit_transform(embeddings)
        score = neighborhood_preservation(embeddings, embedding_3d, k=10)

        print(f"n_neighbors={n_neighbors}, min_dist={min_dist}, score={score:.4f}")

        if score > best_score:
            best_score = score
            best_params = {'n_neighbors': n_neighbors, 'min_dist': min_dist}

print("Best params for local preservation:", best_params)

/usr/local/python/3.12.1/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


n_neighbors=10, min_dist=0.05, score=0.3005


/usr/local/python/3.12.1/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


KeyboardInterrupt: 

In [15]:
embeddings_data = np.load('datasets/bible_embeddings.npz')
embeddings = embeddings_data['embeddings']

['embeddings', 'ids']


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
sample = 'love'
sample_embedding = model.encode([sample])[0]

similarities = model.similarity([sample_embedding], embeddings)

/usr/local/python/3.12.1/lib/python3.12/site-packages/sentence_transformers/util/tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  a = torch.tensor(a)


In [16]:
top10_indices = np.argsort(similarities)[-10:][::-1]
print(top10_indices)
print(similarities[top10_indices])

print(embeddings_data['ids'][top10_indices])

[30614 29274 30610 30622 30613 28800 26716 16593 28799 29268]
[0.500237   0.48663467 0.4765573  0.46783173 0.46740094 0.44444823
 0.4381675  0.4304793  0.4249303  0.4104818 ]
[62004011 49004002 62004007 62004019 62004010 46016024 43015017 20007018
 46016023 49003017]


In [2]:
data1 = pd.read_csv('datasets/bible_data_set.csv')
data2 = pd.read_csv('datasets/t_kjv.csv')

print(len(data1))
print(len(data2))
print(data1.loc[0, 'text'].strip())
print(data2.loc[0, 't'])


31102
31102
In the beginning God created the heaven and the earth.
In the beginning God created the heaven and the earth.


In [3]:
data1['text'] = data2['t']
data1['id'] = data2['id']
data1['book_number'] = data2['b']


In [5]:
print(data1.loc[1, 'text'])

And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.


In [6]:
col_to_move = 'id'
col_data = data1.pop(col_to_move)
data1.insert(0, col_to_move, col_data)

col_to_move = 'book_number'
col_data = data1.pop(col_to_move)
data1.insert(3, col_to_move, col_data)

In [7]:
data1.head()

,id,citation,book,book_number,chapter,verse,text
0,1001001,Genesis 1:1,Genesis,1,1,1,In the beginning God created the heaven and th...
1,1001002,Genesis 1:2,Genesis,1,1,2,"And the earth was without form, and void; and ..."
2,1001003,Genesis 1:3,Genesis,1,1,3,"And God said, Let there be light: and there wa..."
3,1001004,Genesis 1:4,Genesis,1,1,4,"And God saw the light, that it was good: and G..."
4,1001005,Genesis 1:5,Genesis,1,1,5,"And God called the light Day, and the darkness..."


In [8]:
data1.to_csv('datasets/bible_data.csv', index=False)

In [9]:
print(data1[data1['id'] == 1001023]['text'].values)

['And the evening and the morning were the fifth day.']


In [1]:
my_string_ascii = "Genesis 1:1"
byte_size_ascii = len(my_string_ascii.encode('utf-8'))
print(f"The byte size of '{my_string_ascii}' is: {byte_size_ascii} bytes")

The byte size of 'Genesis 1:1' is: 11 bytes
